 # 🚀 Notebook 05: Simulated Trade Execution & Multi-Sleeve Portfolio Engine



 ### Objective:

 Ingests the production execution manifest from Notebook 04, applies Next-Day Close ($T+1$) fills, executes integer share rounding, applies realistic institutional slippage, manages the overlapping 5-sleeve ring buffer, and records mark-to-market accounting.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

from core.paths import TRADES_DIR
from core.settings import TradingConfig
from data_pipeline.loader import load_processed_data

print("=" * 85)
print("🏦 MODULE 05: INSTITUTIONAL TRADE EXECUTION SIMULATOR")
print("=" * 85)

# 1. Ingest Latest Execution Manifest from Notebook 04
manifest_path = TRADES_DIR / "latest_execution_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"❌ Execution manifest {manifest_path} not found! Please run Notebook 04 first."
    )

with open(manifest_path, "r") as f:
    manifest = json.load(f)

orders_parquet = TRADES_DIR / manifest["orders_parquet"]
df_orders = pd.read_parquet(orders_parquet)

decision_date = manifest["decision_date"]
exec_date = manifest["execution_date"]
liquidate_date = manifest["liquidate_sleeve_date"]
holding_period = manifest["holding_period"]
slippage_rate = manifest["slippage_rate"]
benchmark = manifest["benchmark"]

print(f"📅 Decision Date        : {decision_date} (T)")
print(f"🎯 Target Execution Date : {exec_date} (T+1 Close)")
print(
    f"🔄 Liquidate Sleeve Date : {liquidate_date} (Entered H={holding_period} sessions ago)"
)
print(f"💼 Portfolio Equity      : ${manifest['portfolio_equity']:,.2f}")
print(f"📦 Total Orders to Fill  : {len(df_orders)}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output
TRADES_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\trades

🏦 MODULE 05: INSTITUTIONAL TRADE EXECUTION SIMULATOR
📅 Decision Date        : 2026-09-22 (T)
🎯 Target Execution Date : 2026-09-23 (T+1 Close)
🔄 Liquidate Sleeve Date : 2026-09-16 (Entered H=5 sessions ago)
💼 Portfolio Equity      : $100,000.00
📦 Total Orders to Fill  : 22


In [ ]:
# 2. Ingest Market Execution Prices (Day T+1 Close)
data = load_processed_data()
df_ohlcv = data.df_ohlcv

# Check if execution date prices exist in raw OHLCV
all_dates = df_ohlcv.index.get_level_values("Date").unique()
has_realized_prices = pd.Timestamp(exec_date) in all_dates

if has_realized_prices:
    fill_source = f"Realized Market Close on {exec_date}"
    price_slice = df_ohlcv.xs(pd.Timestamp(exec_date), level="Date")
    fill_prices = price_slice["Adj Close"]
else:
    # Forward simulation mode: Use Day T Close as estimated fill price
    fill_source = f"Estimated Pre-Market Close (using latest available close from {decision_date})"
    price_slice = df_ohlcv.xs(pd.Timestamp(decision_date), level="Date")
    fill_prices = price_slice["Adj Close"]

print(f"\n📊 Fill Price Source: {fill_source}")


📊 Fill Price Source: Estimated Pre-Market Close (using latest available close from 2026-09-22)


In [ ]:
# 3. Simulate Integer Share Fills & Execution Slippage
executed_records = []
total_allocated_cash = 0.0
total_slippage_dollars = 0.0

for _, row in df_orders.iterrows():
    tkr = row["Ticker"]
    target_dollars = float(row["Target_Buy_Dollars"])

    if tkr not in fill_prices.index or pd.isna(fill_prices.loc[tkr]):
        raise ValueError(
            f"❌ Execution price for ticker {tkr} not found on {exec_date}!"
        )

    p_exec = float(fill_prices.loc[tkr])

    # Integer share rounding (Institutional Floor)
    shares = int(np.floor(target_dollars / p_exec))
    actual_dollars = shares * p_exec

    # One-way execution slippage = 0.5 * slippage_rate
    slippage_dollars = actual_dollars * (slippage_rate / 2.0)

    total_allocated_cash += actual_dollars
    total_slippage_dollars += slippage_dollars

    executed_records.append(
        {
            "Ticker": tkr,
            "Target_Dollars": target_dollars,
            "Fill_Price": p_exec,
            "Filled_Shares": shares,
            "Executed_Dollars": actual_dollars,
            "Residual_Cash": target_dollars - actual_dollars,
            "Slippage_Dollars": slippage_dollars,
            "Order_Type": "MOC",
            "Action": "BUY",
            "Execution_Date": exec_date,
        }
    )

df_executed = pd.DataFrame(executed_records)

print("\n" + "=" * 85)
print(f"📝 EXECUTED MOC FILL BLOTTER ({exec_date})")
print("=" * 85)
print(
    df_executed[
        [
            "Ticker",
            "Fill_Price",
            "Filled_Shares",
            "Executed_Dollars",
            "Slippage_Dollars",
        ]
    ].to_string(index=False)
)
print("-" * 85)
print(f"  • Total Capital Deployed  : ${total_allocated_cash:,.2f}")
print(
    f"  • Net Slippage Drag       : ${total_slippage_dollars:,.2f} ({(total_slippage_dollars/total_allocated_cash)*10000:.1f} bps)"
)
print(f"  • Unallocated Cash Residue: ${df_executed['Residual_Cash'].sum():,.2f}")
print("=" * 85)


📝 EXECUTED MOC FILL BLOTTER (2026-09-23)
Ticker  Fill_Price  Filled_Shares  Executed_Dollars  Slippage_Dollars
   SPY      773.38              3           2320.14          1.160070
  JAAA       50.68             16            810.88          0.405440
     W      107.35              7            751.45          0.375725
   ALB      116.80              7            817.60          0.408800
    RL      346.06              2            692.12          0.346060
   LEN       83.06             10            830.60          0.415300
   TJX      130.79              6            784.74          0.392370
   IRM      118.13              7            826.91          0.413455
  BMNR       28.76             29            834.04          0.417020
  SNAP        5.62            150            843.00          0.421500
  GDDY       98.24              8            785.92          0.392960
   SNX      283.25              2            566.50          0.283250
   FLR       54.03             15            810

In [ ]:
# 4. Multi-Sleeve Ledger & Ring Buffer State Maintenance
ledger_path = TRADES_DIR / "multi_sleeve_portfolio_ledger.json"

if ledger_path.exists():
    with open(ledger_path, "r") as f:
        ledger = json.load(f)
else:
    ledger = {"active_sleeves": [], "liquidated_history": [], "residual_cash_pool": 0.0}

# Create New Sleeve Record
new_sleeve = {
    "entry_date": exec_date,
    "decision_date": decision_date,
    "positions": df_executed[
        ["Ticker", "Filled_Shares", "Fill_Price", "Executed_Dollars"]
    ].to_dict(orient="records"),
    "total_cost": total_allocated_cash,
    "entry_slippage": total_slippage_dollars,
}

# Check for oldest sleeve to liquidate (held for H=5 days)
liquidated_sleeve = None
active_sleeves = ledger.get("active_sleeves", [])

if len(active_sleeves) >= holding_period:
    liquidated_sleeve = active_sleeves.pop(0)
    # Compute liquidation proceeds
    liq_proceeds = 0.0
    for pos in liquidated_sleeve["positions"]:
        t = pos["Ticker"]
        shs = pos["Filled_Shares"]
        curr_price = float(fill_prices.get(t, pos["Fill_Price"]))
        liq_proceeds += shs * curr_price

    liq_slippage = liq_proceeds * (slippage_rate / 2.0)
    net_liq_cash = liq_proceeds - liq_slippage

    liquidated_sleeve["exit_date"] = exec_date
    liquidated_sleeve["gross_proceeds"] = liq_proceeds
    liquidated_sleeve["exit_slippage"] = liq_slippage
    liquidated_sleeve["net_proceeds"] = net_liq_cash
    liquidated_sleeve["realized_pnl"] = net_liq_cash - liquidated_sleeve["total_cost"]

    ledger["liquidated_history"].append(liquidated_sleeve)

    print("\n" + "=" * 85)
    print(f"🔄 LIQUIDATED OLDEST SLEEVE (Entered: {liquidated_sleeve['entry_date']})")
    print("=" * 85)
    print(f"  • Cost Basis       : ${liquidated_sleeve['total_cost']:,.2f}")
    print(f"  • Gross Proceeds   : ${liq_proceeds:,.2f}")
    print(f"  • Exit Slippage    : ${liq_slippage:,.2f}")
    print(
        f"  • Net Realized PnL : ${liquidated_sleeve['realized_pnl']:,.2f} ({(liquidated_sleeve['realized_pnl']/liquidated_sleeve['total_cost'])*100:.2f}%)"
    )
    print("=" * 85)

# Append new sleeve into active ring buffer
active_sleeves.append(new_sleeve)
ledger["active_sleeves"] = active_sleeves
ledger["residual_cash_pool"] += float(df_executed["Residual_Cash"].sum())

# Persist Ledger to Disk
with open(ledger_path, "w") as f:
    json.dump(ledger, f, indent=2)

df_executed.to_parquet(
    TRADES_DIR / f"executed_fills_{pd.Timestamp(exec_date).strftime('%Y%m%d')}.parquet",
    index=False,
)

print(f"\n💾 Updated Portfolio Ledger : {ledger_path.name}")
print(
    f"📊 Currently Active Sleeves : {len(active_sleeves)} of {holding_period} slots populated"
)
print("=" * 85)
print("✅ SIMULATION COMPLETE: Fills and Multi-Sleeve State Updated.")


💾 Updated Portfolio Ledger : multi_sleeve_portfolio_ledger.json
📊 Currently Active Sleeves : 1 of 5 slots populated
✅ SIMULATION COMPLETE: Fills and Multi-Sleeve State Updated.
